# Tutorial: dependency trees and relation layers

This package has two visualisers for EstNLTK `Text` objects. Both draw SVG arc diagrams
and follow EstNLTK's visualiser conventions: configure once, then call on any number of
texts.

- `DependencyTreeVisualiser` draws the dependency tree of each sentence, with annotation
  rows under the words and boxes over spans.
- `RelationArcVisualiser` draws a `RelationLayer` as arcs between the spans of each
  relation.

Calling a visualiser displays the diagrams in Jupyter; `svg`, `svgs` and `html_output`
return strings.

In [1]:
from estnltk import Text
from estnltk_core import RelationLayer

from estnltk_syntax_vis import DependencyTreeVisualiser, RelationArcVisualiser, Style

## 1. Dependency trees

The visualiser needs a syntax layer with the attributes `id`, `head` (0 for the root) and
`deprel`. Here it is EstNLTK's `maltparser_syntax` (needs Java). The default
`syntax_layer` is `stanza_syntax`, the layer of `StanzaSyntaxTagger`.

In [2]:
text = Text(
    "Eesti Vabariigi president Alar Karis kohtus eile Tallinnas Soome peaministriga. "
    "Kohtumisel arutati Tallinna ja Helsingi vahelist tunnelit."
)
text.tag_layer(["maltparser_syntax", "ner"])

vis = DependencyTreeVisualiser(syntax_layer="maltparser_syntax")
vis(text)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 Eesti 
 Vabariigi 
 president 
 Alar 
 Karis 
 kohtus 
 eile 
 Tallinnas 
 Soome 
 peaministriga 
 . 
 
 
 
 
 
 
 
 
 
 
 
 
 
 root 
 obl 
 obl 
 cc 
 conj 
 amod 
 obj 
 punct 
 Kohtumisel 
 arutati 
 Tallinna 
 ja 
 Helsingi 
 vahelist 
 tunnelit 
 .

### Rows and overlay boxes

`rows` lists annotations to show under the words, as `(layer, attribute)`.
`overlay_layer` draws a box over the words of each span of a layer, labelled with
`overlay_label`.

In [3]:
vis = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech"), ("morph_analysis", "form")],
    overlay_layer="ner",
    overlay_label="nertag",
)
vis(text, sentence=0)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 Eesti 
 H 
 sg g 
 Vabariigi 
 S 
 sg g 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 kohtus 
 V 
 s 
 eile 
 D 
 
 Tallinnas 
 H 
 sg in 
 Soome 
 H 
 sg g 
 peaministriga 
 S 
 sg kom 
 . 
 Z 
 
 LOC 
 PER 
 LOC 
 LOC 
 
 partofspeech 
 form

A row can also be `(label, function)`: the function gets the word's span in the syntax
layer and returns the value to show. For ambiguous layers such as `morph_analysis`, rows
show the first annotation; `ambiguous="all"` shows all distinct values joined with `|`.

In [4]:
DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[
        ("lemma", lambda word: text.morph_analysis.get(word.base_span).lemma[0]),
        ("deprel", lambda word: word.deprel),
    ],
)(text, sentence=1)

root 
 obl 
 obl 
 cc 
 conj 
 amod 
 obj 
 punct 
 Kohtumisel 
 kohtumine 
 obl 
 arutati 
 arutama 
 root 
 Tallinna 
 Tallinn 
 obl 
 ja 
 ja 
 cc 
 Helsingi 
 Helsingi 
 conj 
 vahelist 
 vaheline 
 amod 
 tunnelit 
 tunnel 
 obj 
 . 
 . 
 punct 
 
 lemma 
 deprel

### Highlights and fragments

`highlight` emphasises some words: arcs between them are coloured and arcs to the rest of
the sentence are dashed. It takes a span (e.g. a named entity), several spans, or word
indices together with `sentence=`. Only the sentence with the highlight is drawn.

In [5]:
vis(text, highlight=text.ner[1])

root 
 nmod 
 nmod 
 nsubj 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 appos 
 flat 
 Eesti 
 H 
 sg g 
 Vabariigi 
 S 
 sg g 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 kohtus 
 V 
 s 
 eile 
 D 
 
 Tallinnas 
 H 
 sg in 
 Soome 
 H 
 sg g 
 peaministriga 
 S 
 sg kom 
 . 
 Z 
 
 LOC 
 PER 
 LOC 
 LOC 
 
 partofspeech 
 form

`window=0` keeps only the highlighted words and the heads they attach to; hidden words
become "…". A larger `window` keeps that many words on each side.

In [6]:
vis(text, highlight=text.ner[1], window=0)

appos 
 flat 
 … 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 … 
 PER 
 
 partofspeech 
 form

In [7]:
vis(text, highlight=[text.words[2], text.words[7]], window=0)

root 
 nsubj 
 obl 
 … 
 president 
 S 
 sg n 
 … 
 kohtus 
 V 
 s 
 … 
 Tallinnas 
 H 
 sg in 
 … 
 LOC 
 
 partofspeech 
 form

In [8]:
vis(text, sentence=1, highlight=[2, 3, 4], window=1)

root 
 obl 
 cc 
 conj 
 … 
 arutati 
 V 
 ti 
 Tallinna 
 H 
 sg g 
 ja 
 J 
 
 Helsingi 
 H 
 sg g 
 vahelist 
 A 
 sg p 
 … 
 LOC 
 LOC 
 
 partofspeech 
 form

## 2. Relation layers

The examples use a coreference layer from EstNLTK's RelationLayer tutorial: each relation
links a `mention` to an `entity`.

In [9]:
coref_text = Text(
    'Mari kirjeldas õhinal, kuidas ta väiksena "Sipsikut" luges: "Ma ei suutnud '
    'seda raamatut kohe kuidagi käest ära panna! Nii põnev oli see!"'
).tag_layer(["maltparser_syntax", "clauses"])
coref = RelationLayer(
    "coreference", span_names=["mention", "entity"], text_object=coref_text
)
coref.add_annotation(mention=(30, 32), entity=(0, 4))
coref.add_annotation(mention=(61, 63), entity=(0, 4))
coref.add_annotation(mention=(75, 88), entity=(42, 52))
coref.add_annotation(mention=(133, 136), entity=(42, 52))
coref_text.add_layer(coref)

### As an overlay on the dependency tree

A `RelationLayer` can be the `overlay_layer`. Each named span becomes a box labelled
`name (relation index)`. Overlapping boxes are stacked and their fills add up.

In [10]:
DependencyTreeVisualiser(syntax_layer="maltparser_syntax", overlay_layer="coreference")(
    coref_text, sentence=0
)

root 
 nsubj 
 advmod 
 punct 
 mark 
 nsubj 
 xcomp 
 punct 
 obj 
 punct 
 ccomp 
 punct 
 punct 
 nsubj 
 aux 
 conj 
 det 
 obj 
 advmod 
 advmod 
 obl 
 compound:prt 
 xcomp 
 punct 
 Mari 
 kirjeldas 
 õhinal 
 , 
 kuidas 
 ta 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 : 
 " 
 Ma 
 ei 
 suutnud 
 seda 
 raamatut 
 kohe 
 kuidagi 
 käest 
 ära 
 panna 
 ! 
 entity (1) 
 entity (3) 
 mention (0) 
 entity (0) 
 mention (1) 
 mention (2) 
 entity (2)

### As arcs between spans

`RelationArcVisualiser` takes the layer itself. Each relation is drawn as arcs from its
first filled span to its other spans, labelled with the target span's name. Sentences
linked by relations share one diagram.

In [11]:
RelationArcVisualiser()(coref_text.coreference)

entity 
 entity 
 entity 
 entity 
 Mari 
 kirjeldas 
 õhinal 
 , 
 kuidas 
 ta 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 : 
 " 
 Ma 
 ei 
 suutnud 
 seda 
 raamatut 
 kohe 
 kuidagi 
 käest 
 ära 
 panna 
 ! 
 Nii 
 põnev 
 oli 
 see 
 ! 
 "

`relation=` draws only the given relations, `highlight=` emphasises relations or spans
of any layer, and `window` works as for dependency trees.

In [12]:
RelationArcVisualiser()(coref_text.coreference, relation=coref[3], window=1)

entity 
 … 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 … 
 oli 
 see 
 ! 
 …

Options for the drawing:

- `label`: arc labels from a relation attribute (e.g. `label="rel_type"`), a function, or `None`;
- `hub`: the span name to draw the arcs from; `edges`: a function giving the edges of a relation;
- `span_names=True`: span names in the boxes; `hub_label`: a relation attribute under the hub span;
- `rows`: rows under the words, as in `DependencyTreeVisualiser`;
- `annotations`: span layers drawn as tracks of labelled boxes under the rows;
- `attach="head"` with a `syntax_layer`: arcs attach to the syntactic head of each span.

In [13]:
vis = RelationArcVisualiser(
    span_names=True,
    rows=[("morph_analysis", "partofspeech")],
    annotations=[("clauses", "clause_type")],
    attach="head",
    syntax_layer="maltparser_syntax",
)
vis(coref_text.coreference, highlight=coref[0])

entity 
 
 mention 
 
 entity 
 
 mention 
 
 mention 
 
 mention 
 
 regular 
 
 regular 
 
 regular 
 
 regular 
 
 
 
 
 entity 
 entity 
 entity 
 entity 
 Mari 
 S 
 kirjeldas 
 V 
 õhinal 
 D 
 , 
 Z 
 kuidas 
 D 
 ta 
 P 
 väiksena 
 A 
 " 
 Z 
 Sipsikut 
 S 
 " 
 Z 
 luges 
 V 
 : 
 Z 
 " 
 Z 
 Ma 
 P 
 ei 
 V 
 suutnud 
 V 
 seda 
 P 
 raamatut 
 S 
 kohe 
 D 
 kuidagi 
 D 
 käest 
 S 
 ära 
 D 
 panna 
 V 
 ! 
 Z 
 Nii 
 D 
 põnev 
 A 
 oli 
 V 
 see 
 P 
 ! 
 Z 
 " 
 Z 
 
 partofspeech 
 clauses

## 3. Styling

Colours, font and sizes are fields of `Style`, accepted by both visualisers. Diagrams have
a white background, so they also work with dark themes.

In [14]:
compact = Style(word_font_size=12, level_height=16, leg_gap=6, highlight_arc="#2b8a3e")
DependencyTreeVisualiser(syntax_layer="maltparser_syntax", style=compact)(
    text, highlight=text.ner[1], window=0
)

appos 
 flat 
 … 
 president 
 Alar 
 Karis 
 …

## 4. Output

- `svg(...)`: the SVG string of one diagram (pass `sentence` or `highlight` for texts with
  several sentences);
- `svgs(...)`: a list of SVG strings;
- `html_output(...)`: the HTML that calling the visualiser displays.

All take the same `sentence`, `highlight` and `window` arguments as the call.

In [15]:
svg = vis.svg(coref_text.coreference)
print(svg[:100], "...")

<svg xmlns="http://www.w3.org/2000/svg" width="1438" height="171.5" viewBox="0 0 1438 171.5" font-fa ...


Jupyter removes inline SVG from outputs of notebooks you did not run yourself.
`render_svg_as_img=True` wraps each diagram in an `<img>`, so saved outputs still show
when the notebook is opened elsewhere. It is off by default: GitHub's notebook viewer
blocks such images, and their text cannot be selected.

## 5. Other layer names

All layer names are parameters, e.g. for texts from a database with prefixed layers:

```python
DependencyTreeVisualiser(
    syntax_layer="v172_stanza_syntax",
    sentence_layer="v172_sentences",
    overlay_layer="v171_named_entities",
    overlay_label="nertag",
)
```